## ToyAIKit

In [1]:
import os
from dotenv import load_dotenv
from rag_helper import RAGBase
from openai import OpenAI

from ingest import load_faq_data, build_index

In [2]:
key = os.environ.get("GROQ_API_KEY")
assert key and key != "sk-YOUR_KEY_HERE", "GROQ_API_KEY not loaded!"
print("Key loaded:", key[:8] + "...")

Key loaded: gsk_iwSr...


In [3]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")
index = build_index(documents)

Loaded 1349 documents


In [5]:
def search(query):
   boost_dict = {"question": 2.0, "section": 0.5}
   filter_dict = {"course": "llm-zoomcamp"}

   return index.search(
     query, 
     boost_dict=boost_dict,
     filter_dict=filter_dict,
     num_results=5
  )

### Setup

In [6]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

### Registering the tool

In [7]:
#agent_tools = Tools()
#agent_tools.add_tool(search, search_tool)

In [8]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [9]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [10]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [11]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

### The chat interface and runner

In [12]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="llama-3.3-70b-versatile", client=openai_client)
)

### Running one prompt

In [13]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


/workspaces/llm-zoomcamp-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.3-70b-versatile'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
